# Keyword Scoping From Seed

Use this notebook to sanity-check a proposed topic before briefing or writing content.

It combines:
- Google Ads keyword demand for the entered term and related ideas
- lightweight topic clustering for editorial scoping
- optional comparison against existing Lifeline search coverage in GSC

### Quick start (beginner-friendly)
1. Run the **Setup (run once)** cell.
2. In **Parameters**, enter your seed keyword and any optional extra seed terms.
3. Run the remaining cells from top to bottom.
4. Review the demand tables, cluster summary, and editorial recommendation table.

### Google Ads setup
This notebook expects Google Ads credentials via env vars or `GOOGLE_ADS_CONFIGURATION_FILE`.
If those are missing, the notebook will continue where possible and show a clear setup message.

### Links
- GitHub repo: [github.com/aidanm-lla/lla-data](https://github.com/aidanm-lla/lla-data)
- Open this notebook in Colab: [Keyword Scoping From Seed](https://colab.research.google.com/github/aidanm-lla/lla-data/blob/main/notebooks/15_keyword_scoping_from_seed.ipynb)

### Other notebooks
- [Query Drivers by Page](https://colab.research.google.com/github/aidanm-lla/lla-data/blob/main/notebooks/03_query_drivers_by_page.ipynb)
- [Top Search Queries (Sitewide)](https://colab.research.google.com/github/aidanm-lla/lla-data/blob/main/notebooks/07_top_search_queries_sitewide.ipynb)
- [SXO Page-Query Dynamics](https://colab.research.google.com/github/aidanm-lla/lla-data/blob/main/notebooks/08_sxo_page_query_dynamics.ipynb)


In [ ]:
#@title Setup (run once)
import os
import sys

if "google.colab" in sys.modules:
    from google.colab import auth

    auth.authenticate_user()
    if not os.path.exists("lla-data"):
        !git clone -q https://github.com/aidanm-lla/lla-data.git
    repo = os.path.abspath("lla-data")
    if repo not in sys.path:
        sys.path.insert(0, repo)
    !pip install -U -q "plotly>=6.1.1" "kaleido>=1.2.0" "google-ads>=29.2.0"
else:
    for p in ("..", "../.."):
        ap = os.path.abspath(p)
        if ap not in sys.path:
            sys.path.insert(0, ap)

import pandas as pd
import plotly.express as px
from IPython.display import display
from google.cloud import bigquery

import lifeline_theme
from lla_data import config
from lla_data.bq import build_date_params, default_query_window, get_client, load_sql_template, run_query
from lla_data.google_ads_keywords import (
    GoogleAdsConfigurationError,
    cluster_keywords,
    generate_historical_metrics,
    generate_keyword_ideas,
    label_site_coverage,
    normalize_keyword_text,
    summarize_keyword_clusters,
)

lifeline_theme.inject_fonts()
client = get_client()


In [ ]:
#@title Parameters
SEED_KEYWORD = "natural disasters" #@param {type:"string"}
EXTRA_SEED_KEYWORDS = "bushfires, hurricanes" #@param {type:"string"}
DAYS_BACK = 180 #@param {type:"integer"}
COUNTRY = "Australia" #@param {type:"string"}
LANGUAGE = "English" #@param {type:"string"}
MAX_IDEAS = 100 #@param {type:"integer"}
MIN_AVG_MONTHLY_SEARCHES = 20 #@param {type:"integer"}

window = default_query_window(DAYS_BACK)
seed_keywords = [SEED_KEYWORD.strip()] if SEED_KEYWORD.strip() else []
seed_keywords.extend(
    keyword.strip()
    for keyword in EXTRA_SEED_KEYWORDS.replace("\n", ",").split(",")
    if keyword.strip()
)
seed_keywords = list(dict.fromkeys(seed_keywords))

if not seed_keywords:
    raise ValueError("Enter at least one seed keyword.")

print(f"Project: {config.PROJECT_ID}")
print(f"Search Console dataset: {config.SEARCHCONSOLE_DATASET}")
print(f"Coverage window: {window.start_date} to {window.end_date}")
print(f"Market: {COUNTRY} / {LANGUAGE}")
print(f"Seed keywords: {', '.join(seed_keywords)}")


In [ ]:
# Direct demand for the entered seed keyword(s)
google_ads_error = None

try:
    df_seed_metrics = generate_historical_metrics(
        keywords=seed_keywords,
        location_name=COUNTRY,
        language_name=LANGUAGE,
    )
except GoogleAdsConfigurationError as exc:
    google_ads_error = str(exc)
    print("Google Ads setup is not ready yet.")
    print(google_ads_error)
    df_seed_metrics = pd.DataFrame()
except Exception as exc:
    google_ads_error = str(exc)
    print("Google Ads historical metrics failed.")
    print(google_ads_error)
    df_seed_metrics = pd.DataFrame()

if not df_seed_metrics.empty:
    df_seed_metrics = df_seed_metrics[df_seed_metrics["avg_monthly_searches"].fillna(0) >= MIN_AVG_MONTHLY_SEARCHES].copy()
    display(df_seed_metrics[[
        "keyword",
        "avg_monthly_searches",
        "competition",
        "competition_index",
        "low_top_of_page_bid",
        "high_top_of_page_bid",
    ]].reset_index(drop=True))
else:
    print("No historical metrics returned yet.")


In [ ]:
# Related keyword ideas for editorial scoping
if google_ads_error is None:
    try:
        df_keyword_ideas = generate_keyword_ideas(
            keyword_texts=seed_keywords,
            location_name=COUNTRY,
            language_name=LANGUAGE,
            max_ideas=MAX_IDEAS,
        )
        df_keyword_ideas = df_keyword_ideas[df_keyword_ideas["avg_monthly_searches"].fillna(0) >= MIN_AVG_MONTHLY_SEARCHES].copy()
    except Exception as exc:
        google_ads_error = str(exc)
        print("Google Ads keyword ideas failed.")
        print(google_ads_error)
        df_keyword_ideas = pd.DataFrame()
else:
    df_keyword_ideas = pd.DataFrame()

if not df_keyword_ideas.empty:
    display(df_keyword_ideas[[
        "keyword",
        "avg_monthly_searches",
        "competition",
        "competition_index",
        "close_variants",
    ]].head(30).reset_index(drop=True))
else:
    print("No related keyword ideas returned yet.")


In [ ]:
# Cluster the ideas into lightweight editorial topic groups
frames = [df for df in [df_seed_metrics, df_keyword_ideas] if not df.empty]
if frames:
    df_all_keywords = pd.concat(frames, ignore_index=True)
    df_all_keywords["keyword_normalized"] = df_all_keywords["keyword"].map(normalize_keyword_text)
    df_all_keywords = (
        df_all_keywords.sort_values(["avg_monthly_searches", "keyword"], ascending=[False, True])
        .drop_duplicates(subset=["keyword_normalized"], keep="first")
        .reset_index(drop=True)
    )

    df_clustered = cluster_keywords(df_all_keywords)
    df_cluster_summary = summarize_keyword_clusters(df_clustered)
else:
    df_all_keywords = pd.DataFrame()
    df_clustered = pd.DataFrame()
    df_cluster_summary = pd.DataFrame()

if not df_cluster_summary.empty:
    display(df_cluster_summary.reset_index(drop=True))
else:
    print("No topic clusters available yet.")


In [ ]:
# Optional comparison against existing Lifeline search coverage
if not df_clustered.empty:
    coverage_sql = load_sql_template(
        "sql/notebooks/seo_keyword_site_coverage_by_term.sql",
        project_id=config.PROJECT_ID,
        searchconsole_dataset=config.SEARCHCONSOLE_DATASET,
    )
    coverage_params = build_date_params(window) + [
        bigquery.ArrayQueryParameter("keywords", "STRING", df_clustered["keyword_normalized"].dropna().unique().tolist()),
    ]
    df_site_coverage = run_query(client, coverage_sql, params=coverage_params)
    df_site_coverage["coverage_status"] = df_site_coverage.apply(
        lambda row: label_site_coverage(
            site_clicks=row.get("site_clicks"),
            site_impressions=row.get("site_impressions"),
        ),
        axis=1,
    )

    df_clustered = df_clustered.merge(
        df_site_coverage,
        how="left",
        on="keyword_normalized",
    )
    df_clustered["coverage_status"] = df_clustered["coverage_status"].fillna("not_served")

    coverage_priority = {"already_served": 2, "weakly_served": 1, "not_served": 0}
    df_clustered["coverage_rank"] = df_clustered["coverage_status"].map(coverage_priority).fillna(0)
    cluster_coverage = (
        df_clustered.sort_values(["cluster_id", "coverage_rank"], ascending=[True, False])
        .groupby("cluster_id", as_index=False)
        .agg(cluster_coverage_status=("coverage_status", "first"))
    )
    df_cluster_summary = df_cluster_summary.merge(cluster_coverage, how="left", on="cluster_id")
    display(df_site_coverage.reset_index(drop=True))
else:
    df_site_coverage = pd.DataFrame()
    print("No site coverage lookup ran because there were no keywords to compare.")


In [ ]:
# Chart: cluster demand by representative topic
if not df_cluster_summary.empty:
    fig = px.bar(
        df_cluster_summary.sort_values("cluster_avg_monthly_searches", ascending=True),
        x="cluster_avg_monthly_searches",
        y="cluster_label",
        orientation="h",
        color="recommended_content_shape",
        hover_name="representative_keyword",
        hover_data={
            "keyword_count": True,
            "top_keywords": True,
            "mean_competition_index": ":.1f",
            "cluster_avg_monthly_searches": ":,.0f",
        },
        template="lifeline",
        title=f"Topic Cluster Demand for {', '.join(seed_keywords)} ({window.start_date} to {window.end_date})",
    )
    fig.update_layout(height=max(500, 80 * len(df_cluster_summary)), margin={"l": 280, "r": 40, "t": 80, "b": 40})
    fig.update_xaxes(title="Estimated monthly search demand")
    fig.update_yaxes(title="Topic cluster")
    lifeline_theme.add_lifeline_logo(fig)
    fig.show()
else:
    print("No cluster chart available yet.")


In [ ]:
# Editorial recommendation summary
if not df_cluster_summary.empty:
    recommendation_df = df_cluster_summary.copy()

    def choose_action(row: pd.Series) -> str:
        shape = row.get("recommended_content_shape")
        coverage = row.get("cluster_coverage_status", "not_served")
        if shape == "not enough demand":
            return "deprioritise"
        if shape == "split article set":
            return "brief as multiple articles"
        if coverage == "already_served":
            return "review existing page before creating new content"
        return "brief as single article"

    recommendation_df["editorial_action"] = recommendation_df.apply(choose_action, axis=1)
    display(recommendation_df[[
        "representative_keyword",
        "cluster_avg_monthly_searches",
        "keyword_count",
        "recommended_content_shape",
        "cluster_coverage_status",
        "editorial_action",
        "top_keywords",
    ]].reset_index(drop=True))
else:
    print("No recommendation summary available yet.")
    if google_ads_error:
        print("Next step: configure Google Ads credentials, then rerun the notebook.")
